# Yield Prediction Model Training (State-wise India Dataset)
This notebook uploads the dataset, trains a model, evaluates it, and saves artifacts for backend inference.

## Notes
- Target: `Yield`
- Drops: `Production` (avoid leakage: production = area × yield)
- Split: chronological (train years ≤ 2017, test years > 2017)
- Exports: `yield_model_pipeline.joblib`


In [ ]:
## 1) Install Dependencies

!pip -q install scikit-learn pandas numpy joblib seaborn

# Optional (recommended): XGBoost
!pip -q install xgboost


In [ ]:
## 2) Upload Dataset
# Upload crop_yield.csv here.

import os
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()  # upload crop_yield.csv

if len(uploaded) == 0:
    raise ValueError('No file uploaded. Please upload crop_yield.csv')

csv_name = next(iter(uploaded.keys()))
DATA_PATH = os.path.join('/content', csv_name)
print('Using:', DATA_PATH)


In [ ]:
## 3) Load Dataset

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()


In [ ]:
## 4) Quick Data Checks

df.info()
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False).head(10))
print("\nTop crops:\n", df["Crop"].value_counts().head(10) if "Crop" in df.columns else "Crop col missing")
print("\nYield stats:\n", df["Yield"].describe() if "Yield" in df.columns else "Yield col missing")


In [ ]:
# Basic cleaning
df = df.copy()
df.columns = [c.strip() for c in df.columns]

# Normalize Season formatting
if 'Season' in df.columns:
    df['Season'] = df['Season'].astype(str).str.strip()

# Rename to consistent names
rename_map = {
    'Crop_Year': 'Year',
    'Annual_Rainfall': 'Rainfall',
    'Yield': 'Yield'
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ['Crop', 'Year', 'Season', 'State', 'Area', 'Production', 'Rainfall', 'Fertilizer', 'Pesticide', 'Yield']
missing = [c for c in required_cols if c not in df.columns]
missing


In [ ]:
# Drop rows with missing required values
df_model = df.dropna(subset=[c for c in required_cols if c in df.columns]).copy()

# Ensure numeric types
num_cols = ['Area', 'Production', 'Rainfall', 'Fertilizer', 'Pesticide', 'Yield', 'Year']
for c in num_cols:
    if c in df_model.columns:
        df_model[c] = pd.to_numeric(df_model[c], errors='coerce')
df_model = df_model.dropna(subset=num_cols)

df_model.shape


In [ ]:
# Define features/target
# Avoid leakage by dropping Production (production = area * yield)
target_col = 'Yield'
drop_cols = ['Production']

X = df_model.drop(columns=[target_col] + [c for c in drop_cols if c in df_model.columns])
y = df_model[target_col]

X.columns, y.describe()


In [ ]:
from dataclasses import dataclass
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

@dataclass
class YearTransformer(BaseEstimator, TransformerMixin):
    """Scale year to 0-1 based on training data range."""
    min_year: float = None
    max_year: float = None

    def fit(self, X, y=None):
        years = np.array(X).astype(float).reshape(-1)
        self.min_year = float(np.min(years))
        self.max_year = float(np.max(years))
        return self

    def transform(self, X):
        years = np.array(X).astype(float).reshape(-1)
        denom = (self.max_year - self.min_year) if self.max_year != self.min_year else 1.0
        scaled = (years - self.min_year) / denom
        return scaled.reshape(-1, 1)

def safe_log1p(x):
    x = np.array(x, dtype=float)
    x = np.clip(x, a_min=0, a_max=None)
    return np.log1p(x)


In [ ]:
# Column groups
categorical_cols = [c for c in ['Crop', 'State', 'Season'] if c in X.columns]
year_col = ['Year'] if 'Year' in X.columns else []

# Numeric columns
numeric_log_cols = [c for c in ['Area', 'Fertilizer', 'Pesticide', 'Rainfall'] if c in X.columns]
numeric_other_cols = [c for c in X.columns if c not in categorical_cols + year_col + numeric_log_cols]

categorical_cols, year_col, numeric_log_cols, numeric_other_cols


In [ ]:
# Preprocess pipeline
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

year_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('year_scale', YearTransformer())
])

log_numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(safe_log1p, validate=False))
])

other_numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipe, categorical_cols),
    ('year', year_pipe, year_col),
    ('log_num', log_numeric_pipe, numeric_log_cols),
    ('num', other_numeric_pipe, numeric_other_cols)
], remainder='drop')


In [ ]:
# Chronological split
split_year = 2017
if 'Year' not in df_model.columns:
    raise ValueError('Year column missing after rename. Expected Crop_Year -> Year')

train_mask = df_model['Year'] <= split_year
test_mask = df_model['Year'] > split_year

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]
X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

X_train.shape, X_test.shape


In [ ]:
# Train baseline: RandomForest
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)

rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', rf_model)
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

print('RF R2:', r2_score(y_test, rf_pred))
print('RF MAE:', mean_absolute_error(y_test, rf_pred))


In [ ]:
# Train: XGBoost (usually strongest)
xgb_model = XGBRegressor(
    n_estimators=1200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    random_state=42,
    reg_lambda=1.0,
    n_jobs=-1
)

xgb_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', xgb_model)
])

xgb_pipeline.fit(X_train, y_train)
xgb_pred = xgb_pipeline.predict(X_test)

print('XGB R2:', r2_score(y_test, xgb_pred))
print('XGB MAE:', mean_absolute_error(y_test, xgb_pred))


In [ ]:
## 6) Save Best Pipeline Artifact
from sklearn.utils.validation import check_is_fitted

rf_r2 = r2_score(y_test, rf_pred)
xgb_r2 = r2_score(y_test, xgb_pred)

best_name, best_pipeline = ('xgb', xgb_pipeline) if xgb_r2 >= rf_r2 else ('rf', rf_pipeline)
print('Best:', best_name, '| RF R2:', rf_r2, '| XGB R2:', xgb_r2)

# Guard: if you ran this cell before training, fail with a clear error.
try:
    check_is_fitted(best_pipeline)
except Exception as e:
    raise RuntimeError('Best pipeline is not fitted yet. Run the training cells above first.') from e

import joblib
out_path = '/content/yield_model_pipeline.joblib'
joblib.dump(best_pipeline, out_path)
print('Saved:', out_path)

# Download to your local machine
from google.colab import files
files.download(out_path)
